In [1]:
import numpy as np
import torch
import torch.nn as nn

## Integer Positional Embedding

In [5]:
class IntegerPositionalEmbedding(nn.Module):
    def __init__(self, max_seq_length, embedding_dim, scale=1.0):
        super().__init__()
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.scale = scale
        self.embeddings = nn.Embedding(max_seq_length, embedding_dim)
        
    def forward(self, x):
        seq_length = x.shape[1]
        positions = torch.arange(seq_length, device=x.device)
        pos_embed = self.embeddings(positions) * self.scale
        return x + pos_embed.unsqueeze(0)

batch_size, seq_length, embedding_dim = 2, 10, 64
x = torch.randn(batch_size, seq_length, embedding_dim)

integer_pe = IntegerPositionalEmbedding(max_seq_length=512, embedding_dim=embedding_dim)
output = integer_pe(x)

print("Integer Positional Embedding:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output sample:\n{output[0, :3, :4]}\n")

Integer Positional Embedding:
Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Output sample:
tensor([[ 1.0902,  0.3501, -0.5821,  1.8135],
        [ 3.8552, -2.2378,  2.5556,  3.0254],
        [ 0.4598,  0.4155, -1.7627,  0.3438]], grad_fn=<SliceBackward0>)



## Binary Positional Embedding

In [6]:
class BinaryPositionalEmbedding(nn.Module):
    def __init__(self, max_seq_length, embedding_dim):
        super().__init__()
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        
        self.register_buffer('binary_embeddings', self._compute_binary_embeddings(max_seq_length, embedding_dim))
    
    def _compute_binary_embeddings(self, max_seq_length, embedding_dim):
        binary_embed = torch.zeros(max_seq_length, embedding_dim)
        
        for pos in range(max_seq_length):
            # Convert position to binary
            binary_str = bin(pos)[2:].zfill(embedding_dim)
            for i, bit in enumerate(binary_str):
                binary_embed[pos, i] = float(bit)
        
        return binary_embed
    
    def forward(self, x):
        seq_length = x.shape[1]
        pos_embed = self.binary_embeddings[:seq_length, :x.shape[-1]]
        return x + pos_embed.unsqueeze(0)

binary_pe = BinaryPositionalEmbedding(max_seq_length=512, embedding_dim=embedding_dim)
output = binary_pe(x)

print("Binary Positional Embedding:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Binary embeddings for first 5 positions:\n{binary_pe.binary_embeddings[:5, :8]}\n")

Binary Positional Embedding:
Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Binary embeddings for first 5 positions:
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])



## Sinusoidal Positional Embedding


In [7]:
class SinusoidalPositionalEmbedding(nn.Module):
    """
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, max_seq_length, embedding_dim):
        super().__init__()
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        
        self.register_buffer('pe', self._compute_pe(max_seq_length, embedding_dim))
    
    def _compute_pe(self, max_seq_length, embedding_dim):
        pe = torch.zeros(max_seq_length, embedding_dim)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * 
                           -(np.log(10000.0) / embedding_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        
        if embedding_dim % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        return pe
    
    def forward(self, x):
        seq_length = x.shape[1]
        pos_embed = self.pe[:seq_length, :x.shape[-1]]
        return x + pos_embed.unsqueeze(0)

sinusoidal_pe = SinusoidalPositionalEmbedding(max_seq_length=512, embedding_dim=embedding_dim)
output = sinusoidal_pe(x)

print("Sinusoidal Positional Embedding:")
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Positional embeddings for first 3 positions:\n{sinusoidal_pe.pe[:3, :8]}\n")


Sinusoidal Positional Embedding:
Input shape: torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
Positional embeddings for first 3 positions:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.6816,  0.7318,  0.5332,  0.8460,  0.4093,  0.9124],
        [ 0.9093, -0.4161,  0.9975,  0.0709,  0.9021,  0.4315,  0.7469,  0.6649]])

